# Notebook 3 — Interactive Model Training
Train a single ticker interactively for debugging / hyperparameter exploration.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src/models')
sys.path.insert(0, '../src/evaluation')
from train_lstm_sentiment import build_model
from evaluate import compute_metrics

TICKER = 'AAPL'  # change this to try other tickers
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
data = np.load(f'../processed_data/{TICKER}_sequences.npz', allow_pickle=True)
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

In [ ]:
model = build_model()
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5),
]
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150, batch_size=32,
    callbacks=callbacks, verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('MSE Loss'); axes[0].legend()
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
y_pred = model.predict(X_test).flatten()
metrics = compute_metrics(y_test, y_pred, TICKER, 'LSTM-Sentiment')
print('Test metrics:')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

In [ ]:
dates = data['dates_test']
import pandas as pd
plt.figure(figsize=(14, 5))
plt.plot(dates, y_test, label='Actual', color='#1e40af', linewidth=1.5)
plt.plot(dates, y_pred, label='LSTM-Sentiment Predicted', color='#10b981', linewidth=1.5, linestyle='--')
plt.title(f'{TICKER} — Test Set Predictions')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()